## Desafio 1: Classificação Binária com Perceptron Treinável
Detector preliminar de transações financeiras suspeitas/fraudulentas com base no valor normalizado da transação ($x_1$) e frequência de operações recentes ($x_2$).

In [2]:
import numpy as np

# 1. Dados de Treinamento
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Amostras de Teste
transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])


# 2. Funções do Perceptron
def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=100):
    """Treina o algoritmo Perceptron utilizando a regra de ajuste de Rosenblatt."""
    n_amostras, n_atributos = X.shape

    # Inicialização dos pesos e viés com 1.0 conforme requisito
    w = np.ones(n_atributos, dtype=float)
    b = 1.0

    for epoca in range(epocas):
        erros_epoca = 0
        for i in range(n_amostras):
            # Combinação linear
            z = np.dot(X[i], w) + b
            # Função degrau
            y_pred = 1 if z >= 0 else 0

            # Erro
            erro = y[i] - y_pred

            if erro != 0:
                # Regra de atualização de Rosenblatt
                w += taxa_aprendizado * erro * X[i]
                b += taxa_aprendizado * erro
                erros_epoca += 1

        # Interrupção antecipada caso convirja totalmente
        if erros_epoca == 0:
            break

    return w, b


def predizer_perceptron(x, w, b):
    """Realiza a inferência para uma nova amostra."""
    z = np.dot(x, w) + b
    return 1 if z >= 0 else 0


# 3. Execução e Treinamento
w_calibrado, b_calibrado = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, taxa_aprendizado=0.1, epocas=100
)

# 4. Avaliação das Transações A e B
pred_A = predizer_perceptron(transacao_A, w_calibrado, b_calibrado)
pred_B = predizer_perceptron(transacao_B, w_calibrado, b_calibrado)

rotulos_diagnostico = {0: "Legítima", 1: "Suspeita (Risco de Fraude)"}

# 5. Exibição dos Resultados no Console
print("=== RESULTADOS DESAFIO 1: PERCEPTRON ===")
print(f"Pesos Finais Ajustados (w): {w_calibrado}")
print(f"Viés Final Ajustado (b): {b_calibrado:.4f}\n")

print(
    f"Transação A [5.0, 4.0]: Classe {pred_A} -> Diagnóstico: {rotulos_diagnostico[pred_A]}"
)
print(
    f"Transação B [7.0, 6.5]: Classe {pred_B} -> Diagnóstico: {rotulos_diagnostico[pred_B]}"
)

=== RESULTADOS DESAFIO 1: PERCEPTRON ===
Pesos Finais Ajustados (w): [-5.55111512e-16  4.00000000e-01]
Viés Final Ajustado (b): -1.2000

Transação A [5.0, 4.0]: Classe 1 -> Diagnóstico: Suspeita (Risco de Fraude)
Transação B [7.0, 6.5]: Classe 1 -> Diagnóstico: Suspeita (Risco de Fraude)


## Desafio 2: Predição de Risco de Churn com Classificador KNN
Predição de risco de cancelamento de clientes SaaS com base em dias de inatividade ($x_1$) e chamados críticos abertos ($x_2$), utilizando cálculos de distância vetorizados (Euclidiana e Manhattan).

In [3]:
# 1. Dados de Treinamento
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Clientes sob avaliação
cliente_1 = np.array([13.0, 10.0])  # k=3, distancia="euclidiana"
cliente_2 = np.array([7.0, 1.5])  # k=5, distancia="manhattan"


# 2. Função de Classificação KNN
def classificar_knn(X_treino, y_treino, ponto_consulta, k=3, metrica="euclidiana"):
    """Classifica um novo ponto com base nos k vizinhos mais próximos de forma vetorizada."""
    metrica = metrica.lower()

    # Cálculo Vetorizado das Distâncias
    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum((X_treino - ponto_consulta) ** 2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(X_treino - ponto_consulta), axis=1)
    else:
        raise ValueError(
            "Métrica de distância não suportada. Use 'euclidiana' ou 'manhattan'."
        )

    # Obtenção dos índices dos k vizinhos mais próximos
    indices_vizinhos = np.argsort(distancias)[:k]
    distancias_vizinhos = distancias[indices_vizinhos]
    classes_vizinhos = y_treino[indices_vizinhos]

    # Determinação da classe por maioria simples de votos
    votos, contagens = np.unique(classes_vizinhos, return_counts=True)
    classe_predita = votos[np.argmax(contagens)]

    return classe_predita, indices_vizinhos, distancias_vizinhos


# 3. Execução para os Clientes de Teste
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

# Cliente 1
pred_c1, idx_c1, dist_c1 = classificar_knn(
    X_treino_churn,
    y_treino_churn,
    cliente_1,
    k=3,
    metrica="euclidiana",
)

# Cliente 2
pred_c2, idx_c2, dist_c2 = classificar_knn(
    X_treino_churn,
    y_treino_churn,
    cliente_2,
    k=5,
    metrica="manhattan",
)

# 4. Exibição dos Diagnósticos
print("=== RESULTADOS DESAFIO 2: CLASSIFICADOR KNN ===\n")

print(f"--- CLIENTE 1 [13.0, 10.0] (k=3, Euclidiana) ---")
print(f"Diagnóstico: {rotulos_churn[pred_c1]} (Classe {pred_c1})")
print(f"Índices dos Vizinhos Próximos: {idx_c1}")
print(f"Distâncias dos Vizinhos: {np.round(dist_c1, 4)}\n")

print(f"--- CLIENTE 2 [7.0, 1.5] (k=5, Manhattan) ---")
print(f"Diagnóstico: {rotulos_churn[pred_c2]} (Classe {pred_c2})")
print(f"Índices dos Vizinhos Próximos: {idx_c2}")
print(f"Distâncias dos Vizinhos: {np.round(dist_c2, 4)}")

=== RESULTADOS DESAFIO 2: CLASSIFICADOR KNN ===

--- CLIENTE 1 [13.0, 10.0] (k=3, Euclidiana) ---
Diagnóstico: Alto Risco (Classe 1)
Índices dos Vizinhos Próximos: [4 5 6]
Distâncias dos Vizinhos: [6.3246 7.0711 9.2195]

--- CLIENTE 2 [7.0, 1.5] (k=5, Manhattan) ---
Diagnóstico: Baixo Risco (Classe 0)
Índices dos Vizinhos Próximos: [3 2 1 0 4]
Distâncias dos Vizinhos: [ 1.5  2.5  5.5  5.5 10.5]


## Desafio 3: Recomendação de Servidores Cloud por Similaridade Espacial
Sistema de dimensionamento (sizing) de máquinas virtuais por cálculo de menor distância Euclidiana vetorizada em espaço 3D (vCPUs, RAM, SSD).

In [4]:
# 1. Catálogo de Instâncias e Nomes
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0],
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training",
]

# Perfil Demandado e Parâmetros
demanda_aplicacao = np.array([12.0, 28.0, 450.0])
K_recomendacoes = 2


# 2. Função de Recomendação por Similaridade Espacial
def recomendar_servidores(catalogo, demanda, k=2):
    """Calcula a distância euclidiana vetorizada e retorna os k servidores mais similares."""
    # Distância Euclidiana Vetorizada 3D
    distancias = np.sqrt(np.sum((catalogo - demanda) ** 2, axis=1))

    # Ordenação dos índices da menor para a maior distância
    indices_ordenados = np.argsort(distancias)[:k]

    return indices_ordenados, distancias[indices_ordenados]


# 3. Execução da Recomendação
indices_rec, distancias_rec = recomendar_servidores(
    catalogo_servidores, demanda_aplicacao, k=K_recomendacoes
)

# 4. Relatório Formatado
print("=== RESULTADOS DESAFIO 3: RECOMENDAÇÃO CLOUD ===")
print(
    f"Perfil Demandado: {demanda_aplicacao[0]:.0f} vCPUs | {demanda_aplicacao[1]:.0f} GB RAM | {demanda_aplicacao[2]:.0f} GB SSD\n"
)
print("--- RANKING DE MÁQUINAS RECOMENDADAS ---")

for pos, (idx, dist) in enumerate(zip(indices_rec, distancias_rec), start=1):
    nome = nomes_servidores[idx]
    spec = catalogo_servidores[idx]
    print(f"{pos}º Lugar: {nome}")
    print(
        f"   Especificações: {spec[0]:.0f} vCPUs | {spec[1]:.0f} GB RAM | {spec[2]:.0f} GB SSD"
    )
    print(f"   Distância Geométrica: {dist:.4f}\n")

=== RESULTADOS DESAFIO 3: RECOMENDAÇÃO CLOUD ===
Perfil Demandado: 12 vCPUs | 28 GB RAM | 450 GB SSD

--- RANKING DE MÁQUINAS RECOMENDADAS ---
1º Lugar: Database Enterprise
   Especificações: 16 vCPUs | 32 GB RAM | 500 GB SSD
   Distância Geométrica: 50.3190

2º Lugar: Medium Backend & Cache
   Especificações: 8 vCPUs | 16 GB RAM | 250 GB SSD
   Distância Geométrica: 200.3996

